In [1]:
# LightGBM install (agar nahi hai)
import subprocess
subprocess.run(["pip", "install", "lightgbm"], capture_output=True)

import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

print(f"✅ LightGBM version: {lgb.__version__}")

✅ LightGBM version: 4.6.0


In [2]:
# Load data
with open("../models/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

df = pd.read_csv("../data/upi_features.csv")
X = df[feature_names]
y = df["is_failed"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Failure rate train: {y_train.mean():.2%}")
print(f"Failure rate test:  {y_test.mean():.2%}")

Train: (400000, 33), Test: (100000, 33)
Failure rate train: 29.27%
Failure rate test:  29.27%


In [3]:
# Class imbalance handle karo
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale:.2f}")

# LightGBM model
lgbm_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
)

print("✅ LightGBM training complete!")

Scale pos weight: 2.42
✅ LightGBM training complete!


In [5]:
# LightGBM predictions
y_proba_lgbm = lgbm_model.predict_proba(X_test)[:, 1]
auc_lgbm = roc_auc_score(y_test, y_proba_lgbm)

# Best threshold for LightGBM
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_lgbm)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
best_thresh_lgbm = thresholds[np.argmax(f1_scores)]
best_f1_lgbm = np.max(f1_scores)

y_pred_lgbm = (y_proba_lgbm >= best_thresh_lgbm).astype(int)

# XGBoost comparison (from full data)
with open("../models/xgb_model.pkl", "rb") as f:
    xgb_model = pickle.load(f)

y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
auc_xgb = roc_auc_score(y_test, y_proba_xgb)
y_pred_xgb = (y_proba_xgb >= 0.2782).astype(int)

print("=" * 45)
print(f"{'Metric':<20} {'XGBoost':>10} {'LightGBM':>10}")
print("=" * 45)
print(f"{'AUC-ROC':<20} {auc_xgb:>10.4f} {auc_lgbm:>10.4f}")
print(f"{'Best Threshold':<20} {0.2782:>10.4f} {best_thresh_lgbm:>10.4f}")
print(f"{'F1 Score':<20} {f1_score(y_test, y_pred_xgb):>10.4f} {best_f1_lgbm:>10.4f}")
print(f"{'Recall':<20} {(y_pred_xgb[y_test==1].sum()/y_test.sum()):>10.4f} {(y_pred_lgbm[y_test==1].sum()/y_test.sum()):>10.4f}")
print("=" * 45)

Metric                  XGBoost   LightGBM
AUC-ROC                  0.6246     0.6542
Best Threshold           0.2782     0.4455
F1 Score                 0.4783     0.4904
Recall                   0.7969     0.7497


In [6]:
# Save LightGBM model
with open("../models/lgbm_model.pkl", "wb") as f:
    pickle.dump(lgbm_model, f)

# Save LightGBM threshold
import json
with open("../models/lgbm_threshold_config.json", "w") as f:
    json.dump({"optimal_threshold": float(best_thresh_lgbm)}, f)

print(f"✅ LightGBM model saved!")
print(f"✅ LightGBM threshold saved: {float(best_thresh_lgbm):.4f}")

✅ LightGBM model saved!
✅ LightGBM threshold saved: 0.4455
